In [1]:
import os
import pandas as pd
import random
import numpy as np
from scipy.stats import expon

## Data Augmentation
### Slicing

In [ ]:
def generate_slices(df, col_name):
    sliced_df = pd.DataFrame({'sample_target_idx': np.repeat(df.sample_target_idx.unique(),25),
                              'label': np.repeat(df.label.unique(), 25),
                              'data_type': np.repeat(df.data_type.unique(), 25),
                              'nonempty_data_type_1': np.repeat(df.nonempty_data_type_1.unique(), 25),
                              'cycle_no': range(1,26)
                             })
    for i in range(4):
        sliced_df[f'slice{i}'] = df[col_name][(i*5): (25 + i*5)].values
    return sliced_df


### Shiftinng

In [ ]:
# estimate distribution of each cycle using exponential family
loc = np.zeros(40)
scale = np.zeros(40)
for i in range(40):
    loc[i], scale[i] = expon.fit(data[data.cycle_no == (i+1)].normalized)


def shift_k_cycles(df, loc, scale):
    k = df.k.max()
    series = df.normalized
    truncated_series = series[:(40 - k)]
    added_series = np.zeros(k)
    for i in range(k):
        added_series[i] = expon.rvs(loc[i], scale[i])
    df['shifted_normalized'] = np.concatenate([added_series, truncated_series])
    return df

## GRU